# Combine Synthetic X-ray Checkpoint Datasets

Merges several `synth-dataset-pipeline` checkpoint datasets — each produced by
the SD3 generation notebook's "Publish as a Kaggle Dataset" cell — into one
combined `output/metadata_txt2img.json` + `output/images/txt2img/`.

Pure file merging: no GPU, no repo clone, no model download. Just attach the
datasets as input and run top to bottom.

## Before you run

**Kaggle settings** (right sidebar → Session options):
- **Accelerator:** None / CPU — this notebook does no image generation.
- **Internet:** ON only if you plan to run the optional publish cell at the
  end (it talks to the Kaggle API). Off is fine otherwise.

**Add data:** attach every source dataset listed in the next cell via
*Add Input* (search by its owner/slug, e.g. `whiteflags26/synthetic-xrays-sd3`).

**Kaggle Secrets** (optional, only for the publish cell): `KAGGLE_USERNAME` /
`KAGGLE_KEY`, same as the main generation notebook.

## How the merge works

- Reads every source's `metadata_txt2img.json` (searched for anywhere under
  its mount path, so it doesn't matter whether Kaggle extracted it flat or
  nested inside an `output/` folder) and every `.png` under that same path.
- Combines them **by uid**, processing sources in the exact order you list
  them below. A uid seen for the first time is added as-is; a uid seen again
  in a later source only *replaces* the earlier copy if the later one is more
  complete (every view has an `image_path`, no `error_prompt`) — an already-
  complete uid is never downgraded, regardless of order.
- Copies every image file into one flat `output/images/txt2img/`, skipping
  any filename that's already there (so re-running this notebook is safe).
- Order mostly matters for tie-breaking when the same uid genuinely appears
  in more than one source (e.g. overlapping offset ranges) — reports that
  exist in only one source are included exactly the same regardless of order.

## 1. Sources — in the order you want them combined

Edit this list: paths are exactly as they appear once each dataset is
attached via *Add Input*. If a path is wrong the next cell prints what it
actually found under `/kaggle/input` so you can fix it.

In [ ]:
from pathlib import Path

# Listed in merge order: sd3 -> sd3-2 -> sd3-5 -> sd3-6 -> sd3-3
SOURCES = [
    "/kaggle/input/datasets/whiteflags26/synthetic-xrays-sd3",
    "/kaggle/input/datasets/mirsayad210042136/synthetic-xrays-sd3-2",
    "/kaggle/input/datasets/whiteflags26/synthetic-xrays-sd3-5",
    "/kaggle/input/datasets/sakibhossain323/synthetic-xrays-sd3-6",
    "/kaggle/input/datasets/sakibhossain323/synthetic-xrays-sd3-3",
]

print("Checking sources ...\n")
any_missing = False
for src in SOURCES:
    root = Path(src)
    if not root.exists():
        any_missing = True
        print(f"MISSING  {src}")
        continue
    n_meta = len(list(root.rglob("metadata_txt2img.json")))
    n_png = len(list(root.rglob("*.png")))
    print(f"OK       {src}\n         {n_meta} metadata_txt2img.json file(s), {n_png} .png file(s) found")

if any_missing:
    print("\nOne or more sources are missing. Contents of /kaggle/input:")
    for p in sorted(Path("/kaggle/input").rglob("*")):
        if p.is_dir():
            print(" ", p)
    print("\nFix the paths in SOURCES above to match what's actually mounted, then re-run this cell.")

## 2. Merge

Safe to re-run — copying skips files that already exist, and the metadata
merge is recomputed from scratch each time from the sources, not from a
previous run's output.

In [ ]:
import json
import shutil
from pathlib import Path

OUTPUT_DIR = Path("output")
DEST_IMAGES = OUTPUT_DIR / "images" / "txt2img"
DEST_META = OUTPUT_DIR / "metadata_txt2img.json"
DEST_IMAGES.mkdir(parents=True, exist_ok=True)


def is_complete(entry):
    return (
        not entry.get("error_prompt")
        and entry.get("views")
        and all("image_path" in v for v in entry["views"])
    )


by_uid = {}

for src in SOURCES:
    root = Path(src)
    if not root.exists():
        print(f"⚠ {src}: path does not exist — skipping (see cell above)\n")
        continue

    meta_matches = list(root.rglob("metadata_txt2img.json"))
    if not meta_matches:
        print(f"⚠ {src}: no metadata_txt2img.json found anywhere under this path — skipping\n")
        continue
    if len(meta_matches) > 1:
        print(f"⚠ {src}: found {len(meta_matches)} metadata_txt2img.json files, using the first: {meta_matches[0]}")

    entries = json.loads(meta_matches[0].read_text())

    added = replaced = kept = 0
    for entry in entries:
        uid = entry["uid"]
        if uid not in by_uid:
            by_uid[uid] = entry
            added += 1
        elif is_complete(entry) and not is_complete(by_uid[uid]):
            by_uid[uid] = entry
            replaced += 1
        else:
            kept += 1

    copied = 0
    for img in root.rglob("*.png"):
        target = DEST_IMAGES / img.name
        if not target.exists():
            shutil.copy2(img, target)
            copied += 1

    print(
        f"{src}\n"
        f"   metadata: {len(entries)} uid(s) -> {added} new, {replaced} replaced an incomplete entry, {kept} kept as-is (already complete)\n"
        f"   images:   {copied} new file(s) copied\n"
    )

merged = [by_uid[uid] for uid in sorted(by_uid)]
with open(DEST_META, "w") as f:
    json.dump(merged, f, indent=2, ensure_ascii=False)

done = sum(1 for e in merged if is_complete(e))
total_images = len(list(DEST_IMAGES.glob("*.png")))

print("=== Combined result ===")
print(f"Total uids: {len(merged)}  (complete: {done}, incomplete/failed: {len(merged) - done})")
print(f"Metadata -> {DEST_META}")
print(f"Images   -> {DEST_IMAGES}  ({total_images} file(s))")

## 3. Sanity check (optional)

Cross-checks the merged metadata against the actual image files on disk —
catches a metadata entry pointing at a PNG that never got copied (a source
whose `image_path` pointed somewhere the `rglob` search didn't reach), or a
PNG with no corresponding metadata entry at all.

In [ ]:
import json
from pathlib import Path

merged = json.loads(Path("output/metadata_txt2img.json").read_text())
disk_files = {p.name for p in Path("output/images/txt2img").glob("*.png")}

meta_files = set()
missing_on_disk = []
for entry in merged:
    for view in entry.get("views", []):
        if "image_path" not in view:
            continue
        name = Path(view["image_path"]).name
        meta_files.add(name)
        if name not in disk_files:
            missing_on_disk.append((entry["uid"], name))

orphaned_on_disk = disk_files - meta_files

print(f"Metadata references {len(meta_files)} image file(s).")
print(f"Disk has {len(disk_files)} image file(s).")

if missing_on_disk:
    print(f"\n⚠ {len(missing_on_disk)} file(s) referenced in metadata but missing on disk:")
    for uid, name in missing_on_disk[:20]:
        print(f"   uid={uid}  {name}")
    if len(missing_on_disk) > 20:
        print(f"   ... and {len(missing_on_disk) - 20} more")
else:
    print("No metadata entries point at a missing file.")

if orphaned_on_disk:
    print(f"\n{len(orphaned_on_disk)} file(s) on disk have no metadata entry pointing at them (harmless, just unused):")
    for name in sorted(orphaned_on_disk)[:10]:
        print(f"   {name}")
else:
    print("No orphaned image files.")

## 4. Publish the combined result as a new Kaggle Dataset (optional)

Same publish logic as the main generation notebook's "Publish as a Kaggle
Dataset" cell — `version` first, falling back to `create` on any failure
(covers both a clear "not found" and an older kaggle-cli's bare 403 for a
dataset that doesn't exist yet).

Use this to get one clean combined `DATASET_SLUG` you can point the main
generation notebook's `DATASET_SLUG` (cell 7) at going forward, instead of
juggling five separate ones.

Requires `KAGGLE_USERNAME` / `KAGGLE_KEY` as Kaggle Secrets.

In [ ]:
import json as _json
from pathlib import Path as _Path

try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    KAGGLE_USERNAME = _secrets.get_secret("KAGGLE_USERNAME")
    KAGGLE_KEY = _secrets.get_secret("KAGGLE_KEY")

    _kaggle_dir = _Path.home() / ".kaggle"
    _kaggle_dir.mkdir(exist_ok=True)
    _kaggle_json = _kaggle_dir / "kaggle.json"
    _kaggle_json.write_text(_json.dumps({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}))
    _kaggle_json.chmod(0o600)

    HAS_KAGGLE_API = True
    print(f"Kaggle API credentials written for user '{KAGGLE_USERNAME}'.")
except Exception:
    HAS_KAGGLE_API = False
    print(
        "No KAGGLE_USERNAME/KAGGLE_KEY secrets set — the publish cell below "
        "will be skipped. Set them in Add-ons -> Secrets to enable it."
    )

In [ ]:
import json
import shutil
import subprocess
from pathlib import Path

!pip install -q -U kaggle

# Must be unique to your account. Reuse the same slug across sessions so each
# publish adds a version instead of creating a duplicate dataset.
DATASET_SLUG = "synthetic-xrays-sd3-combined"

if not HAS_KAGGLE_API:
    print("No Kaggle API credentials — see the cell above.")
else:
    publish_dir = Path("kaggle_dataset_publish")
    if publish_dir.exists():
        shutil.rmtree(publish_dir)
    shutil.copytree("output", publish_dir / "output")

    full_slug = f"{KAGGLE_USERNAME}/{DATASET_SLUG}"
    metadata = {
        "title": "Synthetic Chest X-rays (SD3, combined checkpoint)",
        "id": full_slug,
        "licenses": [{"name": "CC0-1.0"}],
    }
    (publish_dir / "dataset-metadata.json").write_text(json.dumps(metadata, indent=2))

    result = subprocess.run(
        ["kaggle", "datasets", "version", "-p", str(publish_dir),
         "-m", "Combined checkpoint", "--dir-mode", "zip"],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        version_error = (result.stdout + result.stderr).strip()
        print(f"`datasets version` failed, trying `datasets create` "
              f"(first publish, or the dataset doesn't exist yet):\n{version_error}\n")
        create_result = subprocess.run(
            ["kaggle", "datasets", "create", "-p", str(publish_dir), "--dir-mode", "zip"],
            capture_output=True, text=True,
        )
        create_error = (create_result.stdout + create_result.stderr).lower()
        if create_result.returncode == 0:
            result = create_result
        elif "already exists" in create_error:
            print(
                "Dataset already exists, so `version` should have worked — the "
                "original error above is the real problem. Common cause: "
                f"KAGGLE_USERNAME ('{KAGGLE_USERNAME}') doesn't match the "
                "account that owns this dataset slug, or KAGGLE_KEY is stale "
                "(generate a fresh token at kaggle.com/settings)."
            )
            result = create_result
        else:
            result = create_result

    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
    else:
        print(f"\nPublished to kaggle.com/datasets/{full_slug}")
        print(f'Point the main generation notebook\'s DATASET_SLUG (cell 7) at '
              f'"{full_slug}" to continue from this combined checkpoint.')

## Troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| A source shows `MISSING` in cell 2 | Not attached, or Kaggle mounted it at a different path than expected | Check the `/kaggle/input` listing the cell prints, fix the path in `SOURCES` |
| `0 metadata_txt2img.json file(s) found` for an attached source | The dataset's zip hasn't finished extracting server-side yet, or it wasn't published by the standard "Publish as a Kaggle Dataset" cell | Wait a minute and re-run cell 2; if it's still 0, check that dataset's own Data Explorer page to see what it actually contains |
| Sanity check shows files "referenced but missing on disk" | The source's `image_path` in its metadata pointed somewhere the `.rglob("*.png")` search didn't cover — unusual layout | Paste the sanity-check output back — the search path may need adjusting for that source |
| Merge counts look too low (fewer uids than expected) | Two sources may genuinely overlap heavily in uid range, so many are "kept as-is" not "added" | Check the per-source `added`/`replaced`/`kept` breakdown printed by the merge cell — `kept` isn't lost data, it means that uid was already complete from an earlier source |
| `403 Client Error: Forbidden` on publish | Outdated `kaggle` CLI misreporting "doesn't exist yet" as a bare 403 | The publish cell already upgrades `kaggle` and retries with `create` on any failure — if it still fails, check `KAGGLE_USERNAME`/`KAGGLE_KEY` are current |